In [0]:
# Quarantine Review

from pyspark.sql import Row
from pyspark.sql import functions as F
from functools import reduce

CATALOG = "insurance_lakehouse"

quarantine_tables = [
    {
        "dataset": "customers",
        "table": f"{CATALOG}.quarantine.quarantine_invalid_customers"
    },
    {
        "dataset": "policies",
        "table": f"{CATALOG}.quarantine.quarantine_invalid_policies"
    },
    {
        "dataset": "claims",
        "table": f"{CATALOG}.quarantine.quarantine_invalid_claims"
    },
    {
        "dataset": "payments",
        "table": f"{CATALOG}.quarantine.quarantine_invalid_payments"
    },
    {
        "dataset": "fraud_indicators",
        "table": f"{CATALOG}.quarantine.quarantine_invalid_fraud_indicators"
    }
]

summary_rows = []
reason_frames = []

for item in quarantine_tables:
    dataset = item["dataset"]
    table = item["table"]

    df = spark.table(table)
    row_count = df.count()

    summary_rows.append(
        Row(
            dataset=dataset,
            quarantine_table=table,
            quarantine_rows=row_count
        )
    )

    if row_count > 0 and "error_reason" in df.columns:
        reason_df = (
            df.groupBy("error_reason")
            .count()
            .withColumn("dataset", F.lit(dataset))
            .select("dataset", "error_reason", "count")
        )
        reason_frames.append(reason_df)

quarantine_summary_df = spark.createDataFrame(summary_rows)

display(quarantine_summary_df)

if reason_frames:
    quarantine_reason_summary_df = reduce(lambda a, b: a.unionByName(b), reason_frames)
    display(quarantine_reason_summary_df.orderBy("dataset", F.desc("count")))
else:
    print("No quarantine records found.")